In [3]:
import itertools
import numpy as np
import matplotlib.pyplot as plt

from scipy.linalg import solve
from scipy.optimize import differential_evolution, shgo

In [ ]:
#grid util
def build_cartesian_grid(axes): #creates a combination of all possible, pre-defined grid points --> Cartesian product grid
    meshes = np.meshgrid(*axes, indexing='ij')
    points = np.stack([m.ravel() for m in meshes], axis=1)
    shape = tuple(len(ax) for ax in axes)
    return points, shape

def flat_index(multi_index, shape):
    return np.ravel_multi_index(tuple(multi_index), shape)

In [ ]:
def f(t, x): 
    return 0

def g(t, x):
    strike_price = 10
    return (x-strike_price)

def K(t, x, zeta_old, zeta_new, min_cost=29, proportional_part = 1.5e-3):
    total_cost = 0
    for i in range(1,len(zeta_old)):
        total_cost += max(proportional_part*x[i]*abs(zeta_new[i]-zeta_old[i]), min_cost)
    return total_cost

def Gamma(t, x, zeta_old, zeta_new):
    return

def admissible_impulse_set(t, x, zeta_old):
    return 

def Solvency_region(t, x, zeta): 
    return 

In [11]:
#grid utilities
def build_cartesian_grid(axes): 
    meshes = np.meshgrid(*axes, indexing='ij')
    points = np.stack([m.ravel() for m in meshes], axis=1)
    shape = tuple(len(ax) for ax in axes)
    return points, shape

def flat_index(multi_index, shape):
    return np.ravel_multi_index(tuple(multi_index), shape)

#geometry of the computational box
def check_if_inside_box(x, axes, tol=1e-12):
    for d, ax in enumerate(axes):
        if(x[d]<ax[0]-tol or x[d]>ax[1]+tol): return False
    return True

def check_if_on_box_boundary(x, axes, tol=1e-12):
    if(not check_if_inside_box(x=x, axes=axes, tol=tol)): return False
    for d, ax in enumerate(axes):
        if(abs(x[d]-ax[0])<=tol or abs(x[d]-ax[-1])<=tol): return True
    return False

def first_exit_point_box(x0, x1, axes, tol=1e-14): #finds first intersection of a segment with the boundary of the box. Assumes x0 is inside and x1 is outside
    v = x1-x0
    alpha_candidates = []

    for d, ax in enumerate(axes):
        if(abs(v[d])<=tol): continue
        if(x1[d]<ax[0]-tol): 
            alpha = (ax[0]-x0[d])/v[d]
            if(0.0 <= alpha and alpha <= 1.0):
                alpha_candidates.append(alpha)
        elif(x1[d]>ax[-1]+tol):
            alpha = (ax[-1]-x0[d])/v[d]
            if(0.0 <= alpha and alpha <= 1.0):
                alpha_candidates.append(alpha)
    
    if(not alpha_candidates):
        return np.array([np.clip(x1[d], axes[d][0], axes[d][-1]) for d in range(len(axes))])
    
    alpha_star = min(alpha_candidates)
    x_exit = x0 + alpha_star*v

    for d, ax in enumerate(axes):
        x_exit[d] = np.clip(x_exit[d], ax[0], ax[-1])
    return x_exit

In [ ]:
#multilinear interpolation
def multilinear_weigths(x, axes): #assumes x is already inside the box
    D = len(axes)
    shape = tuple(len(ax) for ax in axes)

    left_index = []
    lambdas = []

    for d in range(D):
        ax = axes[d]
        xd_clipped = np.clip(x[d], ax[0], ax[-1])

        j = np.searchsorted(ax, xd_clipped, side='right') - 1
        j = max(0, min(j, len(ax)-2))

        x0, x1 = ax[j], ax[j+1]
        lambda_ = 0 if x1==x0 else (xd_clipped-x0)/(x1-x0)

        left_index.append(j)
        lambdas.append(lambda_)
    
    weights = {}

    for bits in itertools.product([0,1], repeat=D):
        index_list = []
        w = 1
        for d, b in enumerate(bits):
            j = left_index[d] + b
            index_list.append(j)
            lambda_ = lambdas[d]
            w *= lambda_ if b else (1.0 - lambda_)
        
        j_flat = flat_index(multi_index=index_list, shape=shape)
        weights[j_flat] = w + weights.get(j_flat, 0.0)

    return weights

def interpolate_inside(V, x, axes):
    w = multilinear_weigths(x=x, axes=axes)
    return sum(weight*V[j] for j, weight in w.items())

def value_or_boundary(V, t, x, axes, g):
    if(check_if_inside_box(x, axes)):
        return interpolate_inside(V=V, x=x, axes=axes)
    
    x_projection = np.array([np.clip(x[d], axes[d][0], axes[d][-1]) for d in range(len(axes))])
    return g(t, x_projection)

#assembling matrix P and vector q
def assemble_P_and_q(axes, t, k, mu, sigma, theta, g, drift_index=None): #MANGLER: bruker ikke theta???
    points, shape = build_cartesian_grid(axes)
    N, D = points.shape 
    M = len(sigma)

    if(drift_index is None):
        drift_index = M-1

    P = np.zeros((N,N), dtype=float)
    qg = np.zeros(N, dtype=float)
    coeff = 1/(2*k**2)

    for i, xi in enumerate(points):
        if(check_if_on_box_boundary(x=xi, axes=axes)):
            continue
        for m, sigma_m in enumerate(sigma):
            drift = (k**2) * mu * xi if m==drift_index else np.zeros(D)
            y_plus = k*sigma_m*xi + drift
            y_minus = -k*sigma_m*xi + drift

            for y_shift in (y_plus, y_minus):
                x_shift = xi + y_shift

                if(check_if_inside_box(x=x_shift, axes=axes)):
                    w = multilinear_weigths(x=x_shift, axes=axes)
                    for j, w_ij in w.items():
                        P[i,j] += coeff*w_ij
                else: 
                    x_exit = first_exit_point_box(x0=xi, x1=x_shift, axes=axes)
                    qg[i] += coeff *g(t, x_exit)
            
            P[i,i] -= coeff
    return P, qg, points

In [13]:
#intervention operator --> using SHGO black-box optimzer for the sup-part
def intervention_value_SHGO(t, x, V_prev, axes, g, Gamma, K, admissible_impulse_set, zeta_bounds, extract_zeta_old):
    #params for SHGO optimiser 
    SHGO_n = 64
    SHGO_iters = 2
    infeasible_penalty = 1e12
    
    zeta_old = extract_zeta_old(x)

    #minimisation objective
    def min_objective(zeta_new):
        zeta_new = np.asarray(zeta_new, dtype=float)

        if(not admissible_impulse_set(t, x, zeta_old, zeta_new)):
            return infeasible_penalty
        
        x_post_intervention = Gamma(t, x, zeta_old, zeta_new)
        value_post_intervention = value_or_boundary(V=V_prev, t=t, x=x_post_intervention, axes=axes, g=g)
        reward = value_post_intervention + K(t, x, zeta_old, zeta_new)

        return -reward #minus here because SHGO is a min-algo., but we are looking for the maximiser
    
    res = shgo(
        min_objective, 
        bounds=zeta_bounds,
        n=SHGO_n, 
        iters=SHGO_iters, 
        sampling_method='sobol'
    )

    if(not np.isfinite(res.fun)):
        raise RuntimeError('SHGO failed to produce a finite intervention value!')
    
    return -res.fun
    
def intervention_vector_SHGO(t, points, V_prev, axes, g, Gamma, K, admissible_impulse_set, zeta_bounds, extract_zeta_old):
    intervention_value = np.zeros(len(points), dtype=float)

    for i, x in enumerate(points):
        intervention_value[i] = intervention_value_SHGO(t=t, x=x, V_prev=V_prev, axes=axes, g=g, Gamma=Gamma, K=K, admissible_impulse_set=admissible_impulse_set, zeta_bounds=zeta_bounds, extract_zeta_old=extract_zeta_old)
    return intervention_value

In [ ]:
def explicit_impulse_step_exit(V_prev, axes, t_prev, dt, theta, )